## Library import

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
import tqdm
import pickle
from time import time_ns

from src.data_import import *
from src.train import train_one_epoch_TC, train_one_epoch
from src.model import EBM

## Definition of constants

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4

IM_SZ_FMN = (1, 28, 28)
IM_SZ_OLI = (1, 64, 64)
IM_SZ_LFW = (1, 128, 128)

N_HEAD = 3
N_EPOCH = 30

print(f"Currently using {DEVICE}")


Currently using cuda


## Data loading

In [3]:
fmn_train_loader, fmn_test_loader = load_fashion_mnist(batch_size=BATCH_SIZE, shuffle=True, class_subset=[9])
oli_train_loader, oli_test_loader = load_olivetti(batch_size=BATCH_SIZE, shuffle=True)
lfw_train_loader, lfw_test_loader, _  = load_lfw(batch_size=BATCH_SIZE, shuffle=True)

In [4]:
from src.sampler import ReplaySampler
from src.information import TotalCorrelationEstimator

## Model training

At each timestep is saved for each epoch a list of dictionaries in the model folder inside a pikle file containing training information. Each epoch-dictioanry has the same structure: 
```text
    traininfo = {
        "e_loss": running_loss / n,
        "e_cd": running_cd / n,
        "e_reg": running_reg / n,
        "e_corr": running_corr / n,
        "e_e_real": running_e_real / n,
        "e_e_fake": running_e_fake / n,
        "l_loss": l_running_loss,
        "l_cd": l_running_cd,
        "l_reg": l_running_reg,
        "l_corr": l_running_corr,
        "l_e_real": l_running_e_real,
        "l_e_fake": l_running_e_fake
    }
```
Where `n` is the length of the `train_loader`. The e_* fields contain the epoch average of said quantity, while the l_* fields contain the list of the actual values per batch. 


In [5]:
def train_full_model(
        model: nn.Module, 
        n_heads: int,
        model_folder: str,
        img_shape: tuple,

        train_loader,

        n_epochs=30,
        learning_rate=1e-3,
        sample_steps=50,
        sample_step_size=10.0,
        noise_std = 0.005,
        energy_reg = 1e-1,
        tc_regulariz = 1e-2,

        device: torch.device = torch.device("cpu"),
    ):

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.000, 0.999))

    sampler = ReplaySampler(model, img_shape=img_shape, buffer_size=600, noise_fraction=0.05, device=device)
    
    total_correlation_estimator = TotalCorrelationEstimator(n_heads, hidden_dim=16, lr=1e-3)
    total_correlation_estimator.to(DEVICE)

    traininfos = []

    for epoch in range(n_epochs+1):
        epoch_loss, traininfo = train_one_epoch_TC(
            model, 
            sampler, 
            train_loader, 
            optimizer,

            sample_steps       = sample_steps,
            sample_step_size   = sample_step_size,
            sample_noise_std   = noise_std,
            energy_reg         = 1e-1,
            tc_regularizations = 1e-2,
            
            tc_estimator=total_correlation_estimator,
            device=device
        )
        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")
        traininfos.append(traininfo)

        if epoch % 5 == 0:
            model_file = f"./models/{model_folder}/EBM_{epoch}_epochs_{img_shape}_shape_{len(model.heads)}_heads_{time_ns()}"
            traininfof = model_file + ".tinfo"
            modelf = model_file + ".pth"
            with open(traininfof, "b+w") as f:
                pickle.dump(traininfos, f)

            with open(modelf, "b+w") as f:
                torch.save(model.state_dict(), f)

In [6]:
model_oli = EBM(image_shape=IM_SZ_OLI, n_heads=N_HEAD).to(DEVICE)
train_full_model(
    model=model_oli,
    n_heads=N_HEAD,
    model_folder="olivetti",
    train_loader=oli_train_loader,
    n_epochs=N_EPOCH,
    img_shape=IM_SZ_OLI,
    device=DEVICE
)

 23%|██▎       | 16/70 [00:05<00:16,  3.18it/s]


KeyboardInterrupt: 

In [ ]:
model_fmn = EBM(image_shape=IM_SZ_FMN, n_heads=N_HEAD).to(DEVICE) 
train_full_model(
    model=model_fmn,
    n_heads=N_HEAD,
    model_folder="fmnist",
    train_loader=fmn_train_loader,
    n_epochs=N_EPOCH,
    img_shape=IM_SZ_FMN,
    device=DEVICE
)

  0%|          | 0/375 [00:00<?, ?it/s]

100%|██████████| 375/375 [01:21<00:00,  4.59it/s]


  CD:     -0.0007
  Reg:    0.0052
  Corr:   0.0012
  E_real: -0.0156
  E_fake: -0.0149
  Gap:    -0.0007
Epoch 1, Loss: 0.0057


100%|██████████| 375/375 [01:20<00:00,  4.66it/s]


  CD:     -0.0051
  Reg:    0.0061
  Corr:   0.0203
  E_real: 0.0270
  E_fake: 0.0321
  Gap:    -0.0051
Epoch 2, Loss: 0.0214


...

  CD:     -0.0014
  Reg:    0.0015
  Corr:   2.2106
  E_real: 0.0003
  E_fake: 0.0017
  Gap:    -0.0014
Epoch 5, Loss: 2.2106


100%|██████████| 375/375 [01:23<00:00,  4.47it/s]


  CD:     -0.0003
  Reg:    0.0008
  Corr:   3.2119
  E_real: 0.0000
  E_fake: 0.0003
  Gap:    -0.0003
Epoch 6, Loss: 3.2124


 61%|██████▏   | 230/375 [00:50<00:32,  4.52it/s]


KeyboardInterrupt: 

In [ ]:
model_lfw = EBM(image_shape=IM_SZ_LFW, n_heads=N_HEAD).to(DEVICE)
train_full_model(
    model=model_lfw,
    n_heads=N_HEAD,
    model_folder="lfw",
    train_loader=lfw_train_loader,
    n_epochs=N_EPOCH,
    img_shape=IM_SZ_LFW,
    device=DEVICE
)